In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔄 Data Mesh & Data as a Product Guide

**Decentralized data architecture with domain-owned data products and self-service analytics**

This guide covers:
- Data mesh principles and architecture
- Domain data ownership
- Data as a product concept
- Data catalogs and governance
- Self-service data platforms
- Data discovery and lineage
- Cross-domain data sharing
- Organizational structure alignment
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Data Mesh Architecture

### Data Mesh Framework
```python
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set
from datetime import datetime
from enum import Enum

class DataDomain(Enum):
    """Organizational data domains"""
    CUSTOMER = "customer"
    PRODUCT = "product"
    FINANCE = "finance"
    OPERATIONS = "operations"
    MARKETING = "marketing"

@dataclass
class DataProduct:
    """Data product owned by domain"""
    product_id: str
    domain: DataDomain
    name: str
    owner: str
    description: str
    schema: Dict
    sla: Dict = field(default_factory=dict)
    created_at: datetime = field(default_factory=datetime.now)
    updated_at: datetime = field(default_factory=datetime.now)
    tags: List[str] = field(default_factory=list)
    quality_score: float = 0.0

@dataclass
class DataProductPort:
    """Port for accessing data product"""
    port_id: str
    product_id: str
    port_type: str  # read, write, stream, batch
    endpoint: str
    authentication: Dict
    rate_limit: int  # requests per minute
    created_at: datetime = field(default_factory=datetime.now)

class DataProductCatalog:
    """Catalog of data products"""
    
    def __init__(self):
        self.products: Dict[str, DataProduct] = {}
        self.ports: Dict[str, DataProductPort] = {}
        self.domain_ownership: Dict[str, List[str]] = {}
    
    def register_product(self, product: DataProduct) -> bool:
        """Register new data product"""
        
        if product.product_id in self.products:
            return False
        
        self.products[product.product_id] = product
        
        # Track domain ownership
        if product.domain.value not in self.domain_ownership:
            self.domain_ownership[product.domain.value] = []
        
        self.domain_ownership[product.domain.value].append(product.product_id)
        
        return True
    
    def register_port(self, port: DataProductPort) -> bool:
        """Register data product port"""
        
        if port.product_id not in self.products:
            return False
        
        self.ports[port.port_id] = port
        return True
    
    def discover_products(self, tags: List[str] = None, domain: str = None) -> List[DataProduct]:
        """Discover data products"""
        
        results = []
        
        for product in self.products.values():
            # Filter by domain
            if domain and product.domain.value != domain:
                continue
            
            # Filter by tags
            if tags and not any(tag in product.tags for tag in tags):
                continue
            
            results.append(product)
        
        return results
    
    def get_product_lineage(self, product_id: str) -> Dict:
        """Get data lineage for product"""
        
        if product_id not in self.products:
            return {}
        
        product = self.products[product_id]
        
        return {
            'product_id': product.product_id,
            'domain': product.domain.value,
            'owner': product.owner,
            'created_at': product.created_at.isoformat(),
            'ports': [p.port_id for p in self.ports.values() if p.product_id == product_id]
        }
    
    def get_domain_products(self, domain: DataDomain) -> List[DataProduct]:
        """Get all products for domain"""
        
        return [p for p in self.products.values() if p.domain == domain]
    
    def get_catalog_stats(self) -> Dict:
        """Get catalog statistics"""
        
        domains_stats = {}
        for domain in DataDomain:
            products = self.get_domain_products(domain)
            domains_stats[domain.value] = {
                'product_count': len(products),
                'avg_quality': sum(p.quality_score for p in products) / len(products) if products else 0
            }
        
        return {
            'total_products': len(self.products),
            'total_ports': len(self.ports),
            'domains': domains_stats
        }
```

### Data Quality & SLA Management
```python
class DataQualityMetrics:
    """Track data quality metrics"""
    
    def __init__(self, product_id: str):
        self.product_id = product_id
        self.completeness_percent = 100.0
        self.accuracy_percent = 100.0
        self.timeliness_minutes = 0
        self.consistency_percent = 100.0
        self.last_updated = datetime.now()
    
    def calculate_quality_score(self) -> float:
        """Calculate overall quality score (0-100)"""
        
        weights = {
            'completeness': 0.25,
            'accuracy': 0.25,
            'timeliness': 0.25,
            'consistency': 0.25
        }
        
        # Timeliness is inverse (lower is better)
        timeliness_score = max(0, 100 - (self.timeliness_minutes / 60 * 10))
        
        score = (
            self.completeness_percent * weights['completeness'] +
            self.accuracy_percent * weights['accuracy'] +
            timeliness_score * weights['timeliness'] +
            self.consistency_percent * weights['consistency']
        )
        
        return min(100.0, max(0.0, score))
    
    def get_quality_report(self) -> Dict:
        """Get quality report"""
        
        return {
            'product_id': self.product_id,
            'completeness_percent': self.completeness_percent,
            'accuracy_percent': self.accuracy_percent,
            'timeliness_minutes': self.timeliness_minutes,
            'consistency_percent': self.consistency_percent,
            'overall_score': self.calculate_quality_score(),
            'last_updated': self.last_updated.isoformat()
        }

class SLATracker:
    """Track SLA compliance"""
    
    def __init__(self, product_id: str, availability_slo: float, latency_slo_ms: int):
        self.product_id = product_id
        self.availability_slo = availability_slo  # e.g., 0.99 for 99%
        self.latency_slo_ms = latency_slo_ms
        self.uptime_hours = 0.0
        self.downtime_hours = 0.0
        self.violations: List[Dict] = []
    
    def check_availability(self, available: bool):
        """Record availability check"""
        
        if available:
            self.uptime_hours += 1.0
        else:
            self.downtime_hours += 1.0
            self.violations.append({
                'timestamp': datetime.now(),
                'type': 'unavailable'
            })
    
    def is_slo_met(self) -> bool:
        """Check if SLO is met"""
        
        total_hours = self.uptime_hours + self.downtime_hours
        
        if total_hours == 0:
            return True
        
        current_availability = self.uptime_hours / total_hours
        
        return current_availability >= self.availability_slo
    
    def get_sla_status(self) -> Dict:
        """Get SLA status"""
        
        total_hours = self.uptime_hours + self.downtime_hours
        current_availability = self.uptime_hours / total_hours if total_hours > 0 else 1.0
        
        return {
            'product_id': self.product_id,
            'availability_slo': self.availability_slo,
            'current_availability': current_availability,
            'slo_met': self.is_slo_met(),
            'uptime_hours': self.uptime_hours,
            'downtime_hours': self.downtime_hours,
            'violations': len(self.violations)
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Domain-Owned Data & Self-Service Platforms

### Self-Service Data Platform
```python
class SelfServicePlatform:
    """Platform for self-service data access"""
    
    def __init__(self):
        self.datasets: Dict[str, Dict] = {}
        self.queries: List[Dict] = []
        self.permissions: Dict[str, List[str]] = {}
    
    def provision_dataset(self, dataset_id: str, source_product_id: str, owner: str) -> bool:
        """Provision dataset from data product"""
        
        self.datasets[dataset_id] = {
            'source_product': source_product_id,
            'owner': owner,
            'created_at': datetime.now(),
            'access_count': 0
        }
        
        return True
    
    def grant_access(self, user: str, dataset_id: str) -> bool:
        """Grant access to dataset"""
        
        if dataset_id not in self.datasets:
            return False
        
        if user not in self.permissions:
            self.permissions[user] = []
        
        if dataset_id not in self.permissions[user]:
            self.permissions[user].append(dataset_id)
        
        return True
    
    def can_access(self, user: str, dataset_id: str) -> bool:
        """Check if user can access dataset"""
        
        return user in self.permissions and dataset_id in self.permissions[user]
    
    def query_dataset(self, user: str, dataset_id: str, query: str) -> Optional[Dict]:
        """Execute query on dataset"""
        
        if not self.can_access(user, dataset_id):
            return None
        
        if dataset_id not in self.datasets:
            return None
        
        self.datasets[dataset_id]['access_count'] += 1
        
        self.queries.append({
            'user': user,
            'dataset_id': dataset_id,
            'query': query,
            'timestamp': datetime.now()
        })
        
        return {
            'dataset_id': dataset_id,
            'query_result': 'mock_result',
            'rows': 100
        }
    
    def get_platform_stats(self) -> Dict:
        """Get platform statistics"""
        
        return {
            'total_datasets': len(self.datasets),
            'total_users': len(self.permissions),
            'total_queries': len(self.queries),
            'most_accessed_dataset': max(
                self.datasets.items(),
                key=lambda x: x[1]['access_count'],
                default=(None, {})
            )[0]
        }
```

### Data Governance & Lineage
```python
class DataLineageTracker:
    """Track data lineage and transformations"""
    
    def __init__(self):
        self.transformations: List[Dict] = []
        self.lineage_graph: Dict[str, List[str]] = {}
    
    def record_transformation(self, source: str, target: str, transform_type: str, metadata: Dict = None):
        """Record data transformation"""
        
        self.transformations.append({
            'source': source,
            'target': target,
            'type': transform_type,
            'timestamp': datetime.now(),
            'metadata': metadata or {}
        })
        
        if source not in self.lineage_graph:
            self.lineage_graph[source] = []
        
        if target not in self.lineage_graph[source]:
            self.lineage_graph[source].append(target)
    
    def get_lineage_path(self, target: str) -> List[str]:
        """Get upstream lineage for target"""
        
        lineage = []
        current = target
        visited = set()
        
        while current and current not in visited:
            lineage.append(current)
            visited.add(current)
            
            # Find upstream
            for source, targets in self.lineage_graph.items():
                if current in targets:
                    current = source
                    break
            else:
                current = None
        
        return lineage
    
    def get_impact_analysis(self, source: str) -> Dict:
        """Get downstream impact of data source change"""
        
        visited = set()
        impacted = []
        
        def traverse(node):
            if node in visited:
                return
            visited.add(node)
            impacted.append(node)
            
            if node in self.lineage_graph:
                for target in self.lineage_graph[node]:
                    traverse(target)
        
        traverse(source)
        
        return {
            'source': source,
            'impacted_targets': impacted,
            'impact_depth': len(impacted) - 1
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Cross-Domain Data Sharing

### Data Contract Framework
```python
@dataclass
class DataContract:
    """Contract between data producer and consumer"""
    contract_id: str
    producer_domain: str
    consumer_domain: str
    data_product_id: str
    schema: Dict
    sla: Dict
    created_at: datetime = field(default_factory=datetime.now)
    status: str = "active"

class DataContractManager:
    """Manage data contracts"""
    
    def __init__(self):
        self.contracts: Dict[str, DataContract] = {}
        self.violations: List[Dict] = []
    
    def create_contract(self, contract: DataContract) -> bool:
        """Create data contract"""
        
        self.contracts[contract.contract_id] = contract
        return True
    
    def validate_contract(self, contract_id: str, data: Dict) -> bool:
        """Validate data against contract"""
        
        if contract_id not in self.contracts:
            return False
        
        contract = self.contracts[contract_id]
        
        # Simple schema validation
        for field, field_type in contract.schema.items():
            if field not in data:
                self.violations.append({
                    'contract_id': contract_id,
                    'violation': f'Missing field: {field}',
                    'timestamp': datetime.now()
                })
                return False
        
        return True
    
    def get_contract_violations(self, contract_id: str) -> List[Dict]:
        """Get violations for contract"""
        
        return [v for v in self.violations if v['contract_id'] == contract_id]
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Data Mesh Checklist

✅ **Architecture**
- [ ] Domains identified
- [ ] Domain ownership assigned
- [ ] Data products designed
- [ ] Ports defined
- [ ] Catalog created

✅ **Self-Service**
- [ ] Self-service platform deployed
- [ ] Dataset discovery enabled
- [ ] Query interface available
- [ ] Access control implemented
- [ ] Monitoring active

✅ **Governance**
- [ ] Data contracts defined
- [ ] Quality metrics tracked
- [ ] SLA compliance measured
- [ ] Lineage tracking enabled
- [ ] Impact analysis available

✅ **Organizational**
- [ ] Domain teams empowered
- [ ] Data ownership clear
- [ ] Training delivered
- [ ] Centers of excellence
- [ ] Communication channels

✅ **Operational**
- [ ] Monitoring dashboards
- [ ] Alerting configured
- [ ] Runbooks documented
- [ ] Incident procedures
- [ ] Team trained
</VSCode.Cell>
```